In [13]:
import sys
print(sys.executable)

d:\python\python.exe


In [14]:
import pandas as pd
import numpy as np
import re
import time
from collections import defaultdict

In [15]:
# ============================================================
# STEP 1: DATA LOAD (memory-optimized)
# ============================================================
print("Loading data...")
t0 = time.time()

s1 = pd.read_csv("Amazon_Dataset/student_resource/dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("Amazon_Dataset/student_resource/dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("Amazon_Dataset/student_resource/dataset/train/train_source3.tsv", sep="\t")

# dtype optimize — RAM bachane ke liye
for df in [s1, s2, s3]:
    df['country'] = df['country'].astype('category')

print(f"Loaded: S1={len(s1)}, S2={len(s2)}, S3={len(s3)} | Time: {time.time()-t0:.1f}s")

Loading data...
Loaded: S1=2206821, S2=5034616, S3=5285603 | Time: 32.9s


In [17]:
import os
import gc
import pickle
import pandas as pd
from collections import Counter

S2_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv"
S3_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked.tsv"

OVERSIZED_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\oversized_keys.pkl"

CHUNK_SIZE = 100_000
BLOCK_LIMIT = 3000

block_counts = Counter()

files = [S2_FILE, S3_FILE]

for file_no, file in enumerate(files, start=1):

    print("\n" + "=" * 60)
    print(f"Processing Source {file_no}")
    print(file)
    print("=" * 60)

    total_rows = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            file,
            sep="\t",
            usecols=["block_key"],
            chunksize=CHUNK_SIZE
        ),
        start=1
    ):

        counts = chunk["block_key"].value_counts()

        for key, count in counts.items():
            block_counts[key] += int(count)

        total_rows += len(chunk)

        print(
            f"Chunk {chunk_no:>4} | "
            f"Rows processed: {total_rows:,} | "
            f"Unique blocks: {len(block_counts):,}"
        )

        del chunk
        del counts
        gc.collect()

print("\n" + "=" * 60)
print("Finding oversized blocks...")
print("=" * 60)

oversized_keys = {
    key
    for key, count in block_counts.items()
    if count > BLOCK_LIMIT
}

print(f"Total unique blocks : {len(block_counts):,}")
print(f"Oversized blocks    : {len(oversized_keys):,}")
print(f"Limit               : {BLOCK_LIMIT:,}")

# SAVE PERMANENTLY
with open(OVERSIZED_FILE, "wb") as f:
    pickle.dump(oversized_keys, f)

print("\n" + "=" * 60)
print("✅ STEP 5 SAVED")
print("=" * 60)
print(f"File: {OVERSIZED_FILE}")
print(f"Size: {os.path.getsize(OVERSIZED_FILE) / 1024:.2f} KB")


Processing Source 1
D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv
Chunk    1 | Rows processed: 100,000 | Unique blocks: 17,844
Chunk    2 | Rows processed: 200,000 | Unique blocks: 27,742
Chunk    3 | Rows processed: 300,000 | Unique blocks: 35,571
Chunk    4 | Rows processed: 400,000 | Unique blocks: 42,122
Chunk    5 | Rows processed: 500,000 | Unique blocks: 48,160
Chunk    6 | Rows processed: 600,000 | Unique blocks: 53,418
Chunk    7 | Rows processed: 700,000 | Unique blocks: 58,433
Chunk    8 | Rows processed: 800,000 | Unique blocks: 63,077
Chunk    9 | Rows processed: 900,000 | Unique blocks: 67,436
Chunk   10 | Rows processed: 1,000,000 | Unique blocks: 71,460
Chunk   11 | Rows processed: 1,100,000 | Unique blocks: 75,357
Chunk   12 | Rows processed: 1,200,000 | Unique blocks: 78,990
Chunk   13 | Rows processed: 1,300,000 | Unique blocks: 82,579
Chunk   14 | Rows processed: 1,400,000 | Unique blocks: 86,000
Chunk   15 | Rows processed: 1,500,

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

TOP_K = 50

S1_FILE = "train_source1_blocked.tsv"

S2_FILE = (
    r"D:\All Data of Desktop with documents\Amazon_Blocking"
    r"\train_source2_blocked.tsv"
)

S3_FILE = (
    r"D:\All Data of Desktop with documents\Amazon_Blocking"
    r"\train_source3_blocked.tsv"
)

print("Loading S1 blocked...")

s1_blocked = pd.read_csv(
    S1_FILE,
    sep="\t"
)

print(
    f"S1 rows: "
    f"{len(s1_blocked):,}"
)

Loading S1 blocked...
S1 rows: 2,206,821


In [19]:
s1_oversized_mask = (
    s1_blocked['block_key']
    .isin(oversized_keys)
)

s1_oversized = s1_blocked.loc[
    s1_oversized_mask,
    [
        'entity_id',
        'block_key',
        'name_clean'
    ]
].copy()

print(
    f"S1 oversized records: "
    f"{len(s1_oversized):,}"
)

S1 oversized records: 1,360,544


In [21]:
import pickle

# kaunse blocks poore ho chuke hain, ye results se nikal lo
done_ids = set(results_oversized.keys())
done_keys = {
    k for k, g in s1_oversized.groupby('block_key')['entity_id']
    if set(g.astype(str)) <= done_ids
}
print(len(done_keys), "blocks done,", len(results_oversized), "results")

with open("step6_ckpt.pkl", "wb") as f:
    pickle.dump({"results": results_oversized, "done_keys": done_keys}, f)

275 blocks done, 859459 results


In [ ]:
import os, pickle, time, gc
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

CKPT = "step6_ckpt.pkl"
BATCH = 10          # ek baar me kitne blocks. Memory kam pade to 5, zyada ho to 20

# ---------- checkpoint save / load ----------
def save_ckpt():
    with open(CKPT + ".tmp", "wb") as f:
        pickle.dump({"results": results_oversized, "done_keys": done_keys}, f)
    os.replace(CKPT + ".tmp", CKPT)

if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        ck = pickle.load(f)
    results_oversized = ck["results"]
    done_keys = ck["done_keys"]
    print(f"Resumed from file: {len(done_keys)} blocks done")

elif 'results_oversized' in globals() and len(results_oversized) > 0:
    # purana cell interrupt kiya tha, memory me jo results hain wo bacha lo
    done_ids = set(results_oversized.keys())
    done_keys = {
        k for k, g in s1_oversized.groupby('block_key')['entity_id']
        if set(g.astype(str)) <= done_ids
    }
    print(f"Resumed from memory: {len(done_keys)} blocks done")

else:
    results_oversized = {}
    done_keys = set()
    print("Fresh start")

# ---------- ek block ka kaam ----------
def process_block(key, cand_parts):
    s1_sub = s1_oversized[s1_oversized['block_key'] == key].copy()
    if len(s1_sub) == 0 or not cand_parts:
        return

    cand_sub = pd.concat(cand_parts, ignore_index=True)

    s1_text = s1_sub['name_clean'].fillna('').astype(str)
    cand_text = cand_sub['name_clean'].fillna('').astype(str)

    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        max_features=5000
    )
    vectorizer.fit(pd.concat([s1_text, cand_text], ignore_index=True))

    s1_vec = vectorizer.transform(s1_text)
    cand_vec = vectorizer.transform(cand_text)

    nn = NearestNeighbors(
        n_neighbors=min(TOP_K, len(cand_sub)),
        metric='cosine',
        algorithm='brute'
    )
    nn.fit(cand_vec)
    distances, indices = nn.kneighbors(s1_vec)

    cand_ids = cand_sub['entity_id'].astype(str).values
    for i, entity_id in enumerate(s1_sub['entity_id']):
        results_oversized[str(entity_id)] = ','.join(cand_ids[indices[i]])

    print(f"  S1: {len(s1_sub):,} | Candidates: {len(cand_sub):,}")

    del (s1_sub, cand_sub, s1_text, cand_text, vectorizer,
         s1_vec, cand_vec, nn, distances, indices, cand_ids)
    gc.collect()

# ---------- main loop ----------
t0 = time.time()
pending = [k for k in oversized_keys if k not in done_keys]
print(f"Pending blocks: {len(pending)} / {len(oversized_keys)}")

try:
    for b in range(0, len(pending), BATCH):
        batch_keys = pending[b:b + BATCH]
        key_set = set(batch_keys)
        batch_parts = {k: [] for k in batch_keys}

        # S2 + S3 ek baar padho, poore batch ke candidates nikal lo
        for file in [S2_FILE, S3_FILE]:
            for chunk in pd.read_csv(
                file,
                sep="\t",
                usecols=['entity_id', 'block_key', 'name_clean'],
                chunksize=CHUNK_SIZE
            ):
                chunk = chunk[chunk['block_key'].isin(key_set)]
                if chunk.empty:
                    continue
                for k, g in chunk.groupby('block_key'):
                    batch_parts[k].append(g)

        # har block process karo
        for key in batch_keys:
            print(f"Block: {key}")
            process_block(key, batch_parts.pop(key))
            done_keys.add(key)

        save_ckpt()   # har batch ke baad save
        print(f"Saved. Done {len(done_keys)}/{len(oversized_keys)} | "
              f"Results: {len(results_oversized):,} | "
              f"{time.time() - t0:.0f}s")
        gc.collect()

finally:
    save_ckpt()       # Interrupt ya error pe bhi progress save

print("\nSTEP 6 COMPLETED")
print(f"Results: {len(results_oversized):,}")
print(f"Time: {time.time() - t0:.1f}s")

In [20]:
results_oversized = {}

t0 = time.time()

for block_no, key in enumerate(
    oversized_keys,
    start=1
):

    print(
        f"\n[{block_no}/{len(oversized_keys)}]"
        f" Processing: {key}"
    )

    s1_sub = s1_oversized[
        s1_oversized['block_key'] == key
    ].copy()

    if len(s1_sub) == 0:
        continue

    print(
        f"S1 records: "
        f"{len(s1_sub):,}"
    )

    cand_parts = []

    # S2 + S3
    for file in [
        S2_FILE,
        S3_FILE
    ]:

        print(f"Reading: {file}")

        for chunk in pd.read_csv(
            file,
            sep="\t",
            usecols=[
                'entity_id',
                'block_key',
                'name_clean'
            ],
            chunksize=CHUNK_SIZE
        ):

            part = chunk[
                chunk['block_key'] == key
            ]

            if len(part) > 0:
                cand_parts.append(
                    part.copy()
                )

            del part
            del chunk

            gc.collect()

    if not cand_parts:

        print("No candidates.")
        continue

    cand_sub = pd.concat(
        cand_parts,
        ignore_index=True
    )

    del cand_parts
    gc.collect()

    print(
        f"Candidates: "
        f"{len(cand_sub):,}"
    )

    # ----------------------------------------
    # TF-IDF
    # ----------------------------------------

    s1_text = (
        s1_sub['name_clean']
        .fillna('')
        .astype(str)
    )

    cand_text = (
        cand_sub['name_clean']
        .fillna('')
        .astype(str)
    )

    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        max_features=5000
    )

    all_text = pd.concat(
        [
            s1_text,
            cand_text
        ],
        ignore_index=True
    )

    vectorizer.fit(all_text)

    s1_vec = vectorizer.transform(
        s1_text
    )

    cand_vec = vectorizer.transform(
        cand_text
    )

    # ----------------------------------------
    # Nearest neighbors
    # ----------------------------------------

    n_neighbors = min(
        TOP_K,
        len(cand_sub)
    )

    nn = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric='cosine',
        algorithm='brute'
    )

    nn.fit(cand_vec)

    distances, indices = nn.kneighbors(
        s1_vec
    )

    # ----------------------------------------
    # Store candidates
    # ----------------------------------------

    cand_ids = (
        cand_sub['entity_id']
        .astype(str)
        .values
    )

    for i, entity_id in enumerate(
        s1_sub['entity_id']
    ):

        candidate_ids = cand_ids[
            indices[i]
        ]

        results_oversized[
            str(entity_id)
        ] = ','.join(
            candidate_ids
        )

    print(
        f"Generated: "
        f"{len(s1_sub):,}"
    )

    # Free memory
    del (
        s1_text,
        cand_text,
        all_text,
        vectorizer,
        s1_vec,
        cand_vec,
        nn,
        distances,
        indices,
        cand_ids,
        cand_sub,
        s1_sub
    )

    gc.collect()

print(
    "\nSTEP 6 COMPLETED"
)

print(
    f"Results: "
    f"{len(results_oversized):,}"
)

print(
    f"Time: "
    f"{time.time() - t0:.1f}s"
)


[1/495] Processing: US_beck
S1 records: 763
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked.tsv
Candidates: 3,242
Generated: 763

[2/495] Processing: US_arts
S1 records: 1,775
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked.tsv
Candidates: 7,050
Generated: 1,775

[3/495] Processing: India_gree
S1 records: 4,052
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked.tsv
Candidates: 12,119
Generated: 4,052

[4/495] Processing: US_glob
S1 records: 5,270
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv
Reading: D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked

KeyboardInterrupt: 

In [ ]:
STEP6_FILE = (
    r"B:\Amazon_Blocking"
    r"\step6_results.pkl"
)

with open(
    STEP6_FILE,
    "wb"
) as f:

    pickle.dump(
        results_oversized,
        f
    )

print("STEP 6 SAVED:")
print(STEP6_FILE)

In [ ]:
if 'candidate_entity_ids' not in s1_blocked.columns:
    s1_blocked['candidate_entity_ids'] = ''

s1_blocked.loc[
    s1_oversized_mask,
    'candidate_entity_ids'
] = (
    s1_blocked.loc[
        s1_oversized_mask,
        'entity_id'
    ]
    .astype(str)
    .map(
        lambda eid:
        results_oversized.get(
            eid,
            ''
        )
    )
)

In [ ]:
candidates_df = s1_blocked[
    [
        'entity_id',
        'candidate_entity_ids'
    ]
].rename(
    columns={
        'entity_id':
        'source1_entity_id'
    }
)

candidates_df[
    'candidate_entity_ids'
] = (
    candidates_df[
        'candidate_entity_ids'
    ].fillna('')
)

print(
    "Final candidates_df shape:",
    candidates_df.shape
)

print(
    candidates_df.head()
)

In [ ]:
FINAL_FILE = (
    r"B:\Amazon_Blocking"
    r"\candidate_pairs.tsv"
)

candidates_df.to_csv(
    FINAL_FILE,
    sep="\t",
    index=False
)

print("\nFINAL FILE SAVED:")
print(FINAL_FILE)